# 6 · Execution and impact

What happens between the decision and the fill.

A historical print tells you the price, never what your order would have
done to it. Here the same seed runs twice, once with your orders and once
without, so every fill is priced against the market where you never traded.

That counterfactual is what arrival price, VWAP and fitted impact models
approximate. Here it is measured directly.

In [1]:
import tradefloor as tf

universe = tf.Universe.random(20, seed=111)

execution = tf.tca.analyse(tf.baselines.Momentum(),
                           seed=7, universe=universe, days=3)

print("fills           :", len(execution.fills))
print("partial fills   :", len(execution.partial_fills()))
print("shortfall (bps) :", round(execution.shortfall_bps(), 3))

fills           : 158
partial fills   : 0
shortfall (bps) : 6.04


## The counterfactual run

`actual_final` is the closing price of each instrument. `baseline_final` is
the same market with the agent's orders removed: same seed, same draws,
everything else identical. Where they differ, that difference is the
agent's footprint.

In [2]:
tickers = [i.ticker for i in universe]
actual, baseline = execution.actual_final, execution.baseline_final

print(f"{'ticker':8s} {'with orders':>13s} {'without':>13s} {'difference':>12s}")
for t, a, b in zip(tickers, actual, baseline):
    if a != b:
        print(f"{t:8s} {a:13.4f} {b:13.4f} {a - b:12.4f}")

untouched = sum(1 for a, b in zip(actual, baseline) if a == b)
print(f"\n{untouched} of {len(tickers)} names were not moved at all.")

ticker     with orders       without   difference
AAA           194.1200      194.9500      -0.8300
AAB            11.7200       11.7400      -0.0200
AAC            22.9700       22.9800      -0.0100
AAD             6.3400        6.3500      -0.0100
AAE            18.9400       18.8800       0.0600
AAG           278.3100      278.4500      -0.1400
AAJ           126.8700      126.8200       0.0500
AAL            55.2900       55.2500       0.0400
AAM           101.5600      101.5700      -0.0100
AAN             4.0400        4.0500      -0.0100
AAO            10.4000       10.4300      -0.0300
AAP           138.1000      145.2200      -7.1200
AAR             4.6800        4.6400       0.0400
AAT           494.9800      494.9600       0.0200

6 of 20 names were not moved at all.


## Which names moved

`moved()` reports every instrument whose final price differs between the two
runs, and it reports that difference in basis points, not in currency. The
column printed below as `price moved` is therefore a bps figure, so
it agrees with `impact_bps` to the rounding: both are the same measurement.
Bps is what lets a displacement on AAR, trading below 5, sit in the same
column as one on AAT, trading near 495. For the move in currency, read the
table above.

In [3]:
moved = execution.moved()
worst = sorted(moved.items(), key=lambda kv: -abs(kv[1]))[:6]

print(f"{'ticker':8s} {'price moved':>13s} {'impact (bps)':>14s}")
for ticker, delta in worst:
    print(f"{ticker:8s} {delta:13.4f} {execution.impact_bps(ticker):14.3f}")

ticker     price moved   impact (bps)
AAP          -490.2906       -490.291
AAR            86.2069         86.207
AAA           -42.5750        -42.575
AAE            31.7797         31.780
AAO           -28.7632        -28.763
AAN           -24.6914        -24.691


## Where the cost fell

`by_step` attributes the shortfall across decision points and `by_ticker`
across names. Positive is cost paid. A negative step, like step 7 below, is
one where the fills came in better than the untraded market: trading back
into your own impact gives some of it back. Cost concentrated in a few steps
says something about the schedule.

In [4]:
steps = execution.by_step()
print("by step:")
for step, value in steps:
    bar = "#" * int(min(40, abs(value) / max(1, max(abs(v) for _, v in steps)) * 40))
    print(f"  step {step:3d} {value:12,.0f}  {bar}")

by step:
  step   6          815  ##################################
  step   7           95  ####
  step   8          451  ###################
  step   9          322  #############
  step  10           72  ###
  step  11          464  ###################
  step  12          515  #####################
  step  13          225  #########
  step  14          547  #######################
  step  15          439  ##################
  step  16          943  ########################################
  step  17          267  ###########


In [5]:
by_ticker = execution.by_ticker()
top = sorted(by_ticker.items(), key=lambda kv: -abs(kv[1]))[:6]
print("largest contributions by name:")
for ticker, value in top:
    print(f"  {ticker:8s} {value:12,.0f}")

largest contributions by name:
  AAH               656
  AAS               591
  AAO               572
  AAI               544
  AAP               510
  AAC               431


## Partial fills

An order asking for more than the book holds at a price does not silently
receive it. Queue position and depth decide what you actually get.

In [6]:
if execution.partial_fills():
    print(f"{len(execution.partial_fills())} partial fills")
    for f in execution.partial_fills()[:5]:
        print("  ", f)
else:
    print("No partial fills at this size; the book absorbed every order.")
    print("Raising participation or size is how you find the edge of that;")
    print("the whole point is that the edge exists and is measurable.")

No partial fills at this size; the book absorbed every order.
Raising participation or size is how you find the edge of that;
the whole point is that the edge exists and is measurable.


## The same orders in a thinner book

The cell above says the edge exists and is measurable, and then does not
reach it: at this size the book absorbs everything. Raising participation
is one way to find the edge. Taking the depth away is the other, and it is
the one that matches what happens to you rather than what you chose.

A scenario is how you say that. `market.liquidity` scales the `avg_volume`
column the market maker quotes off, so every ladder level thins and the
same order walks further up the book. `macro.vix` widens the quote at the
same time, which is what a funding event does to both at once.

Nothing about the agent changes. Same seed, same market, same decisions --
only what it costs to act on them.


In [7]:
squeeze = (tf.Scenario(name="funding squeeze")
           .shock("market.liquidity", operation="multiply", value=0.35,
                  at=0, duration=3)
           .shock("macro.vix", operation="multiply", value=2.0,
                  at=0, duration=3))

print(f"{'book':10s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for label, scenario in (("calm", None), ("squeeze", squeeze)):
    ex = tf.tca.analyse(tf.baselines.Momentum(), seed=7,
                        universe=universe, days=3, scenario=scenario)
    print(f"{label:10s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")


book        shortfall bps   fills  partials


calm                6.040     158         0


squeeze            10.807     148         2


The shortfall roughly doubles, and the partial fills appear: in the thin
book the agent asks for what it asked for before and does not get it. That
is the whole reason this simulator prices execution rather than assuming
it. An evaluation that reads only the price series scores the two runs the
same, because the *prices* barely move -- the shock is in the book.

`tradefloor scenario list` names the scenarios that ship with the package;
`liquidity_crisis` is this shape on a longer horizon, with an assumed
credit response beside it. Load one with `tf.Scenario.load("...")`.

Its `transmission` block is worth reading before believing: those entries
are assumptions its author made, not effects this simulator derives.


## Comparing three algorithms

Same market, same seed, with the counterfactual computed for each.

In [8]:
candidates = {
    "momentum":       tf.baselines.Momentum(),
    "mean reversion": tf.baselines.MeanReversion(),
    "buy and hold":   tf.baselines.BuyAndHold(),
}

print(f"{'algorithm':16s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for name, agent in candidates.items():
    ex = tf.tca.analyse(agent, seed=7, universe=universe, days=3)
    print(f"{name:16s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")

algorithm         shortfall bps   fills  partials
momentum                  6.040     158         0


mean reversion           -1.513     157         0
buy and hold             13.191      20         0


## Provenance

An execution result carries the seed and model fingerprint, so a TCA number
can be cited.

In [9]:
print("seed             :", execution.seed)
print("model fingerprint:", execution.model_fingerprint)

seed             : 7
model fingerprint: pt-v14


## Caveats

**Volume changes were this notebook's structural caveat, and are not any
more.** A ceiling in the engine capped a name's volume response at a four
percent daily move, so a violent day traded like a quiet one. Be precise
about the horizon that cost, because it was not a one-year failure:
`volume_change_acf1` was already inside its 252-day band on `pt-v10`.
Notebook 04 scores that preset at 13 of 14 at the certified horizon and this
is not the row it gives up. What it missed was the tighter band re-derived
at 504 days, so the gap had been rewritten around the horizon
rather than dropped. `pt-v12`, the preset fingerprinted above, raised the
ceiling from four percent to twelve; the statistic came inside at BOTH
horizons and the volume-change gap was retired from the realism envelope.
Schedule-shape conclusions no longer carry that warning at either horizon,
and depth, queue and impact conclusions are sound as they were before.
Notebook 04 prints the panel this rests on. What the envelope still forbids
lives in five other gaps -- horizon, decay-shape, scenario-magnitude,
macro-range, roster-concentration -- none of them about the fill mechanics
measured here.

**Single venue, no latency.** One book per name, orders arrive instantly,
and no strategic counterparties adapt to you.

Full documentation: <https://simoncoombes.github.io/tradefloor/>